In [1]:
# Install required libraries
!pip install -q langchain langchain-community faiss-cpu sentence-transformers transformers gradio
!pip install -q langchain-text-splitters nltk
!pip install -q pytesseract pillow
!apt-get install -y tesseract-ocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.


In [4]:
# Upload your text file
from google.colab import files
uploaded = files.upload()

# Check uploaded files
import os
os.listdir("/content")

Saving rag_data.txt to rag_data.txt


['.config', 'rag_data.txt', 'sample_data']

In [5]:
# Load text document
from langchain_community.document_loaders import TextLoader

loader = TextLoader("/content/rag_data.txt")  # change if needed
documents = loader.load()

# Preview content
print(documents[0].page_content[:500])

Title: Introduction to Artificial Intelligence and Machine Learning

Artificial Intelligence (AI) refers to the ability of machines to perform tasks that normally require human intelligence. These tasks include learning, reasoning, problem-solving, perception, and language understanding.

Machine Learning (ML) is a subset of AI that focuses on building systems that learn from data. Instead of being explicitly programmed, ML models improve their performance as they are exposed to more data.

Ther


In [7]:
# Import splitter
from langchain_text_splitters import NLTKTextSplitter
import nltk

# Download tokenizer
nltk.download('punkt')
nltk.download('punkt_tab')

# Split document into chunks
text_splitter = NLTKTextSplitter(chunk_size=500)
docs = text_splitter.split_documents(documents)

# Preview chunks
print("Number of chunks:", len(docs))
print(docs[0].page_content)

Number of chunks: 8
Title: Introduction to Artificial Intelligence and Machine Learning

Artificial Intelligence (AI) refers to the ability of machines to perform tasks that normally require human intelligence.

These tasks include learning, reasoning, problem-solving, perception, and language understanding.

Machine Learning (ML) is a subset of AI that focuses on building systems that learn from data.

Instead of being explicitly programmed, ML models improve their performance as they are exposed to more data.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [8]:
# Create embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Test embedding
vector = embeddings.embed_query(docs[0].page_content)
print("Embedding vector length:", len(vector))

/tmp/ipykernel_574/2460647262.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding vector length: 384


In [9]:
# Store embeddings in FAISS
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(docs, embeddings)

# Test retrieval
query = "What is Machine Learning?"
results = db.similarity_search(query, k=3)

print("Top retrieved chunk:")
print(results[0].page_content)

Top retrieved chunk:
Title: Introduction to Artificial Intelligence and Machine Learning

Artificial Intelligence (AI) refers to the ability of machines to perform tasks that normally require human intelligence.

These tasks include learning, reasoning, problem-solving, perception, and language understanding.

Machine Learning (ML) is a subset of AI that focuses on building systems that learn from data.

Instead of being explicitly programmed, ML models improve their performance as they are exposed to more data.


In [10]:
# Load text generation model
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="distilgpt2",
    pad_token_id=50256
)

# Test generation
prompt = "Explain Machine Learning in simple terms:\n\n" + results[0].page_content
response = generator(prompt, max_new_tokens=120)[0]["generated_text"]

print(response)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain Machine Learning in simple terms:

Title: Introduction to Artificial Intelligence and Machine Learning

Artificial Intelligence (AI) refers to the ability of machines to perform tasks that normally require human intelligence.

These tasks include learning, reasoning, problem-solving, perception, and language understanding.

Machine Learning (ML) is a subset of AI that focuses on building systems that learn from data.

Instead of being explicitly programmed, ML models improve their performance as they are exposed to more data.
The ML model provides a number of possible methods for machine learning.
Machine Learning (ML) is the language you use to model machine learning.
Machine Learning (ML) is a subset of AI that focuses on building systems that learn from the data.
This is one of the most well known ML models.
Machine Learning (ML) is a subset of AI that focuses on building systems that learn from the data.
Machine Learning (ML) is a subset of AI that focuses on building syste

In [11]:
# History-aware RAG system (clean + limited)

chat_history = []

def ask_rag(query):
    global chat_history

    # Retrieve top 2 relevant chunks
    relevant_chunks = db.similarity_search(query, k=2)
    context = "\n\n".join([chunk.page_content for chunk in relevant_chunks])

    # Keep last 2 interactions only
    history_text = ""
    for h in chat_history[-2:]:
        history_text += h + "\n"

    # Build prompt
    prompt = f"{history_text}\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"

    # Generate response
    output = generator(prompt, max_new_tokens=100)[0]["generated_text"]

    # Clean output
    answer = output.split("Answer:")[-1].strip()

    # Save history
    chat_history.append(f"Q: {query}\nA: {answer}")

    return answer

In [12]:
print(ask_rag("Explain AI in simple terms"))
print(ask_rag("What are types of machine learning?"))
print(ask_rag("What is reinforcement learning?"))

Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I mean, I think I


Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Can you learn from your training and learn from your training? Question: Can you learn from your training and learn from your training? Question: Can you learn from your training and learn from your training? Question: Can you learn from your training and learn from your training? Question: Can you learn from your training and learn from your training? Question: Can you learn from your training and learn from your
It's a new way to learn from machine learning and machine learning.
So, how big is the problem with reinforcement learning?
A: It's a very big problem that is difficult to solve in real-time.
You can learn a bit from the machine learning problem that you have written to you from scratch and ask questions about it as well as other examples.
The problem is that the problem is a problem which has been solved much further in the past.
If you can't


In [13]:
# Simple user interface
import gradio as gr

def rag_interface(query):
    return ask_rag(query)

gr.Interface(
    fn=rag_interface,
    inputs=gr.Textbox(placeholder="Ask something..."),
    outputs="text",
    title="RAG Chatbot",
    description="Ask questions based on your uploaded document"
).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5ac5222a480c2a762e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [14]:
# OCR: Extract text from image
import pytesseract
from PIL import Image

def load_image_text(image_path):
    image = Image.open(image_path)
    text = pytesseract.image_to_string(image)
    return text

In [15]:
# Upload image for OCR
from google.colab import files
uploaded = files.upload()

# Example usage:
# text = load_image_text("/content/your_image.png")
# print(text)

Saving Screenshot 2024-02-08 165616.png to Screenshot 2024-02-08 165616.png
